<a href="https://www.kaggle.com/code/chiro999/transformerpreset?scriptVersionId=306317014" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [3]:
import os

print("Datasets mounted at /kaggle/input:")
print(os.listdir("/kaggle/input"))
print(os.listdir("/kaggle/input/datasets")[:50])
print(os.listdir("/kaggle/input/datasets/organizations")[:100])

Datasets mounted at /kaggle/input:
['datasets']
['organizations']
['nih-chest-xrays']


In [4]:
import os
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays"
print(os.listdir(DATA_ROOT)[:200])

['data']


In [5]:
import os
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

first_png = None
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.endswith(".png"):
            first_png = os.path.join(root, f)
            break
    if first_png:
        break

print("First PNG found:", first_png)


First PNG found: /kaggle/input/datasets/organizations/nih-chest-xrays/data/images_003/images/00006199_010.png


In [6]:
!nvidia-smi

Thu Feb 26 08:11:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!pip -q install timm

import os, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from torchvision import transforms
from sklearn.metrics import roc_auc_score

In [9]:
DATA_ROOT = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
CSV_PATH = DATA_ROOT + "/Data_Entry_2017.csv"

LABELS = [
    "Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule",
    "Pneumonia","Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis",
    "Pleural_Thickening","Hernia"
]
label2idx = {l:i for i,l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [10]:
img_dirs = [
    os.path.join(DATA_ROOT, d, "images")
    for d in os.listdir(DATA_ROOT)
    if d.startswith("images_") and os.path.isdir(os.path.join(DATA_ROOT, d, "images"))
]

print("Image folders found:", len(img_dirs))
print("Example dirs:", img_dirs[:3])

name2path = {}
for d in img_dirs:
    for f in os.listdir(d):
        if f.endswith(".png"):
            name2path[f] = os.path.join(d, f)

print("Indexed images:", len(name2path))
print("Has 00000001_000.png?", "00000001_000.png" in name2path)

Image folders found: 12
Example dirs: ['/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_003/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_012/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_009/images']
Indexed images: 112120
Has 00000001_000.png? True


In [11]:
df = pd.read_csv(CSV_PATH)

patients = df["Patient ID"].unique()
np.random.shuffle(patients)

n = len(patients)
train_ids = set(patients[: int(0.8*n)])
val_ids   = set(patients[int(0.8*n): int(0.9*n)])
test_ids  = set(patients[int(0.9*n):])

train_df = df[df["Patient ID"].isin(train_ids)].copy()
val_df   = df[df["Patient ID"].isin(val_ids)].copy()
test_df  = df[df["Patient ID"].isin(test_ids)].copy()

print(len(train_df), len(val_df), len(test_df))

89703 11221 11196


In [13]:
class ChestXray14(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        fname = row["Image Index"]
        img_path = name2path[fname]  # <-- key fix

        img = Image.open(img_path).convert("L")
        img = Image.merge("RGB", (img, img, img))

        y = np.zeros(NUM_LABELS, dtype=np.float32)
        for f in str(row["Finding Labels"]).split("|"):
            if f in label2idx:
                y[label2idx[f]] = 1.0

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(y)

IMG_SIZE = 224
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.25]*3),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.25]*3),
])

In [14]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(ChestXray14(train_df, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)

val_loader = DataLoader(ChestXray14(val_df, val_tf),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

test_loader = DataLoader(ChestXray14(test_df, val_tf),
                         batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

In [15]:
def compute_pos_weight(df_):
    counts = np.zeros(NUM_LABELS, dtype=np.float64)
    for labels in df_["Finding Labels"].astype(str).values:
        for f in labels.split("|"):
            if f in label2idx:
                counts[label2idx[f]] += 1

    N = len(df_)
    pos = np.clip(counts, 1.0, None)
    neg = N - counts
    return torch.tensor(neg / pos, dtype=torch.float32)

pos_weight = compute_pos_weight(train_df).to(device)
print("pos_weight:", pos_weight)

pos_weight: tensor([  8.7609,  39.6448,   7.5310,   4.5925,  18.7889,  16.7630,  76.3969,
         19.5694,  23.1332,  48.3959,  42.3767,  63.4418,  32.6344, 466.2031],
       device='cuda:0')


In [20]:
MODEL_NAME = "vit_base_patch16_224"  # if OOM: use "vit_small_patch16_224" and/or reduce batch size
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_LABELS).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.05)
scaler = torch.amp.GradScaler("cuda", enabled=(device=="cuda"))

def train_one_epoch():
    model.train()
    total = 0.0
    for x, y in tqdm(train_loader, leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(device=="cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * x.size(0)
    return total / len(train_loader.dataset)

@torch.no_grad()
def eval_auc_per_class(model, loader, device, label_names):
    """Returns (mean_auc, per_class_aucs_list) aligned with label_names."""
    model.eval()
    all_logits, all_y = [], []

    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        logits = model(x).float().cpu()
        all_logits.append(logits)
        all_y.append(y.cpu())

    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_y).numpy()
    y_prob = 1 / (1 + np.exp(-logits))  # sigmoid

    per_class = []
    for i in range(len(label_names)):
        if len(np.unique(y_true[:, i])) < 2:
            per_class.append(np.nan)
        else:
            per_class.append(roc_auc_score(y_true[:, i], y_prob[:, i]))

    mean_auc = float(np.nanmean(per_class))
    return mean_auc, per_class

In [21]:
EPOCHS = 5
best_val = -1
best_path = "/kaggle/working/best_custom_vit_chestxray14.pt"

for epoch in range(1, EPOCHS+1):
    tr_loss = train_one_epoch()
    val_mean_auc, _ = eval_auc_per_class(model, val_loader, device, LABELS)

    print(f"Epoch {epoch}/{EPOCHS} | train loss {tr_loss:.4f} | val mean AUROC {val_mean_auc:.4f}")

    if val_mean_auc > best_val:
        best_val = val_mean_auc
        torch.save(model.state_dict(), best_path)
        print("✅ saved best:", best_path)

print("Best val mean AUROC:", best_val)

Epoch 1/5 | train loss 1.2462 | val mean AUROC 0.7403
✅ saved best: /kaggle/working/best_custom_vit_chestxray14.pt


Epoch 2/5 | train loss 1.1040 | val mean AUROC 0.7889
✅ saved best: /kaggle/working/best_custom_vit_chestxray14.pt


Epoch 3/5 | train loss 1.0392 | val mean AUROC 0.8031
✅ saved best: /kaggle/working/best_custom_vit_chestxray14.pt


Epoch 4/5 | train loss 1.0076 | val mean AUROC 0.8089
✅ saved best: /kaggle/working/best_custom_vit_chestxray14.pt


Epoch 5/5 | train loss 0.9645 | val mean AUROC 0.8025
Best val mean AUROC: 0.8089425364613455


In [22]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_mean_auc, test_aucs = eval_auc_per_class(model, test_loader, device, LABELS)

print("Test mean AUROC:", test_mean_auc)
for name, auc in zip(LABELS, test_aucs):
    print(f"{name:18s}: {auc:.4f}")


Test mean AUROC: 0.8067230532124942
Atelectasis       : 0.7868
Cardiomegaly      : 0.8900
Effusion          : 0.8670
Infiltration      : 0.6987
Mass              : 0.8062
Nodule            : 0.7316
Pneumonia         : 0.7628
Pneumothorax      : 0.8455
Consolidation     : 0.7911
Edema             : 0.8775
Emphysema         : 0.8725
Fibrosis          : 0.7690
Pleural_Thickening: 0.7778
Hernia            : 0.8177
